In [1]:
# Temporal HRV Modelling

#This notebook extends the CHF HRV project from independent-window baseline models to patient-level temporal modelling.

#The objective is to preserve the chronological progression of HRV windows for each patient and investigate whether temporal information can improve patient-level risk prediction.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import os

features_path = "../data/processed/features"

print("Current working directory:")
print(os.getcwd())

print("\nFeature directory:")
print(os.path.abspath(features_path))

print("\nFiles:")
for file in os.listdir(features_path):
    print(file)

Current working directory:
C:\ddata\git program\chf-hrv-risk-prediction\notebooks

Feature directory:
C:\ddata\git program\chf-hrv-risk-prediction\data\processed\features

Files:
chf201_hrv_features.csv
hrv_features.csv
ml_dataset.csv
patient_metadata.csv
patient_rf_oof.csv
rf_window_oof.csv


In [4]:
ml_dataset = pd.read_csv(
    "../data/processed/features/ml_dataset.csv"
)

print("Shape:", ml_dataset.shape)

print("\nColumns:")
print(ml_dataset.columns.tolist())

print("\nFirst 5 rows:")
display(ml_dataset.head())

Shape: (11109, 17)

Columns:
['patient_id', 'mean_rr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf', 'window_id', 'rr_count', 'sdnn_trend', 'rmssd_trend', 'pnn50_trend', 'age', 'sex', 'nyha_class', 'risk_label']

First 5 rows:


,patient_id,mean_rr,sdnn,rmssd,pnn50,lf_power,hf_power,lf_hf,window_id,rr_count,sdnn_trend,rmssd_trend,pnn50_trend,age,sex,nyha_class,risk_label
0,chf01,901.686747,38.977235,39.595538,1.208459,32.602446,27.739686,1.175300,0,332,0.122392,0.2208,0.044787,71.0,M,III-IV,1
1,chf01,920.773006,24.194927,12.849903,0.000000,28.498706,24.485597,1.163897,1,326,0.122392,0.2208,0.044787,71.0,M,III-IV,1
2,chf01,983.383607,40.970951,62.164468,0.986842,125.582722,674.823706,0.186097,2,305,0.122392,0.2208,0.044787,71.0,M,III-IV,1
3,chf01,987.671053,33.713954,32.012374,6.270627,70.207401,155.640244,0.451088,3,304,0.122392,0.2208,0.044787,71.0,M,III-IV,1
4,chf01,986.473684,30.488855,25.534032,2.970297,58.816251,75.394149,0.780117,4,304,0.122392,0.2208,0.044787,71.0,M,III-IV,1


In [5]:
print("Number of patients:",
      ml_dataset["patient_id"].nunique())

print("Number of windows:",
      len(ml_dataset))

print("\nWindows per patient:")
print(
    ml_dataset
    .groupby("patient_id")
    .size()
    .describe()
)

Number of patients: 44
Number of windows: 11109

Windows per patient:
count     44.000000
mean     252.477273
std       22.463913
min      196.000000
25%      239.750000
50%      248.000000
75%      273.000000
max      288.000000
dtype: float64


In [6]:
print(
    ml_dataset[
        ["patient_id", "window_id", "risk_label"]
    ].head(20)
)
print("\nIs window_id numeric?")
print(
    ml_dataset["window_id"].dtype
)

   patient_id  window_id  risk_label
0       chf01          0           1
1       chf01          1           1
2       chf01          2           1
3       chf01          3           1
4       chf01          4           1
5       chf01          5           1
6       chf01          6           1
7       chf01          7           1
8       chf01          8           1
9       chf01          9           1
10      chf01         10           1
11      chf01         11           1
12      chf01         12           1
13      chf01         13           1
14      chf01         14           1
15      chf01         15           1
16      chf01         16           1
17      chf01         17           1
18      chf01         18           1
19      chf01         19           1

Is window_id numeric?
int64


In [7]:
ordering_check = (
    ml_dataset
    .groupby("patient_id")["window_id"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print(ordering_check)

print(
    "\nPatients with ordered windows:",
    ordering_check.sum(),
    "/",
    len(ordering_check)
)

patient_id
chf01     True
chf02     True
chf03     True
chf04     True
chf05     True
chf06     True
chf07     True
chf08     True
chf09     True
chf10     True
chf11     True
chf12     True
chf13     True
chf14     True
chf15     True
chf201    True
chf202    True
chf203    True
chf204    True
chf205    True
chf206    True
chf207    True
chf208    True
chf209    True
chf210    True
chf211    True
chf212    True
chf213    True
chf214    True
chf215    True
chf216    True
chf217    True
chf218    True
chf219    True
chf220    True
chf221    True
chf222    True
chf223    True
chf224    True
chf225    True
chf226    True
chf227    True
chf228    True
chf229    True
Name: window_id, dtype: bool

Patients with ordered windows: 44 / 44


In [8]:
label_check = (
    ml_dataset
    .groupby("patient_id")["risk_label"]
    .nunique()
)

print(
    "Patients with one unique risk label:",
    (label_check == 1).sum()
)

print(
    "Patients with multiple risk labels:",
    (label_check > 1).sum()
)

print("\nLabel counts per patient:")
print(label_check.value_counts())

Patients with one unique risk label: 44
Patients with multiple risk labels: 0

Label counts per patient:
risk_label
1    44
Name: count, dtype: int64


In [9]:
print("Risk label distribution:")
print(ml_dataset["risk_label"].value_counts())

print("\nRisk label by NYHA class:")
print(
    ml_dataset
    .groupby("nyha_class")["risk_label"]
    .value_counts()
)
print("Risk label distribution:")
print(ml_dataset["risk_label"].value_counts())

print("\nRisk label by NYHA class:")
print(
    ml_dataset
    .groupby("nyha_class")["risk_label"]
    .value_counts()
)

Risk label distribution:
risk_label
1    7960
0    3149
Name: count, dtype: int64

Risk label by NYHA class:
nyha_class  risk_label
I           0             1089
II          0             2060
III         1             4380
III-IV      1             3580
Name: count, dtype: int64
Risk label distribution:
risk_label
1    7960
0    3149
Name: count, dtype: int64

Risk label by NYHA class:
nyha_class  risk_label
I           0             1089
II          0             2060
III         1             4380
III-IV      1             3580
Name: count, dtype: int64


In [10]:
label_check = (
    ml_dataset
    .groupby("patient_id")["risk_label"]
    .nunique()
)

print(
    "Patients with one unique risk label:",
    (label_check == 1).sum()
)

print(
    "Patients with multiple risk labels:",
    (label_check > 1).sum()
)

print("\nLabel counts per patient:")
print(label_check.value_counts())

Patients with one unique risk label: 44
Patients with multiple risk labels: 0

Label counts per patient:
risk_label
1    44
Name: count, dtype: int64


In [11]:
patient_labels = (
    ml_dataset[
        ["patient_id", "nyha_class", "risk_label"]
    ]
    .drop_duplicates()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

print("Number of patients:", len(patient_labels))

print("\nPatient-level class distribution:")
print(
    patient_labels["risk_label"]
    .value_counts()
    .sort_index()
)

print("\nPatient-level NYHA distribution:")
print(
    patient_labels["nyha_class"]
    .value_counts()
)

print("\nPatient-level label mapping:")
display(patient_labels)

Number of patients: 44

Patient-level class distribution:
risk_label
0    12
1    32
Name: count, dtype: int64

Patient-level NYHA distribution:
nyha_class
III       17
III-IV    15
II         8
I          4
Name: count, dtype: int64

Patient-level label mapping:


,patient_id,nyha_class,risk_label
0,chf01,III-IV,1
1,chf02,III-IV,1
2,chf03,III-IV,1
3,chf04,III-IV,1
4,chf05,III-IV,1
5,chf06,III-IV,1
6,chf07,III-IV,1
7,chf08,III-IV,1
8,chf09,III-IV,1
9,chf10,III-IV,1


In [12]:
temporal_features = [
    "mean_rr",
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_power",
    "hf_power",
    "lf_hf",
    "rr_count",
    "sdnn_trend",
    "rmssd_trend",
    "pnn50_trend"
]

print("Temporal features:")
for feature in temporal_features:
    print(feature)

print("\nNumber of temporal features:",
      len(temporal_features))

Temporal features:
mean_rr
sdnn
rmssd
pnn50
lf_power
hf_power
lf_hf
rr_count
sdnn_trend
rmssd_trend
pnn50_trend

Number of temporal features: 11


In [13]:
sequence_length = 32

sequence_counts = (
    ml_dataset
    .groupby("patient_id")
    .size()
    .apply(lambda n: n // sequence_length)
)

print("Sequence length:", sequence_length)

print("\nComplete 32-window sequences per patient:")
print(sequence_counts.describe())

print("\nPatients with number of complete sequences:")
print(sequence_counts.value_counts().sort_index())

Sequence length: 32

Complete 32-window sequences per patient:
count    44.000000
mean      7.386364
std       0.654711
min       6.000000
25%       7.000000
50%       7.000000
75%       8.000000
max       9.000000
dtype: float64

Patients with number of complete sequences:
6     3
7    22
8    18
9     1
Name: count, dtype: int64


In [14]:
sequence_length = 32

sequence_data = []

for patient_id, patient_df in ml_dataset.groupby("patient_id"):

    patient_df = patient_df.sort_values("window_id").reset_index(drop=True)

    patient_label = patient_df["risk_label"].iloc[0]

    n_complete = len(patient_df) // sequence_length

    for seq_idx in range(n_complete):

        start = seq_idx * sequence_length
        end = start + sequence_length

        sequence = patient_df.iloc[start:end]

        sequence_data.append({
            "patient_id": patient_id,
            "sequence_id": seq_idx,
            "risk_label": patient_label,
            "X": sequence[temporal_features].values
        })

print("Total sequences:", len(sequence_data))

print(
    "Unique patients:",
    len(set(item["patient_id"] for item in sequence_data))
)

print("\nFirst sequence shape:")
print(sequence_data[0]["X"].shape)

print("\nFirst sequence:")
print(
    sequence_data[0]["patient_id"],
    sequence_data[0]["sequence_id"],
    sequence_data[0]["risk_label"]
)

Total sequences: 325
Unique patients: 44

First sequence shape:
(32, 11)

First sequence:
chf01 0 1


In [15]:
X_sequences = np.array([
    item["X"]
    for item in sequence_data
])

y_sequences = np.array([
    item["risk_label"]
    for item in sequence_data
])

patient_sequences = np.array([
    item["patient_id"]
    for item in sequence_data
])

print("X_sequences shape:", X_sequences.shape)
print("y_sequences shape:", y_sequences.shape)
print("patient_sequences shape:", patient_sequences.shape)

print("\nSequence class distribution:")
print(
    pd.Series(y_sequences).value_counts().sort_index()
)

print("\nUnique patients:")
print(
    len(np.unique(patient_sequences))
)

X_sequences shape: (325, 32, 11)
y_sequences shape: (325,)
patient_sequences shape: (325,)

Sequence class distribution:
0     91
1    234
Name: count, dtype: int64

Unique patients:
44


In [16]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

sequence_folds = list(
    group_kfold.split(
        X_sequences,
        y_sequences,
        groups=patient_sequences
    )
)

print("Number of folds:", len(sequence_folds))

for fold_number, (train_idx, val_idx) in enumerate(
    sequence_folds,
    start=1
):

    train_patients = set(
        patient_sequences[train_idx]
    )

    val_patients = set(
        patient_sequences[val_idx]
    )

    overlap = train_patients.intersection(
        val_patients
    )

    print(
        f"\nFold {fold_number}"
    )

    print(
        "Training sequences:",
        len(train_idx)
    )

    print(
        "Validation sequences:",
        len(val_idx)
    )

    print(
        "Training patients:",
        len(train_patients)
    )

    print(
        "Validation patients:",
        len(val_patients)
    )

    print(
        "Patient overlap:",
        len(overlap)
    )

Number of folds: 5

Fold 1
Training sequences: 258
Validation sequences: 67
Training patients: 35
Validation patients: 9
Patient overlap: 0

Fold 2
Training sequences: 259
Validation sequences: 66
Training patients: 35
Validation patients: 9
Patient overlap: 0

Fold 3
Training sequences: 259
Validation sequences: 66
Training patients: 35
Validation patients: 9
Patient overlap: 0

Fold 4
Training sequences: 259
Validation sequences: 66
Training patients: 35
Validation patients: 9
Patient overlap: 0

Fold 5
Training sequences: 265
Validation sequences: 60
Training patients: 36
Validation patients: 8
Patient overlap: 0


In [17]:
for fold_number, (train_idx, val_idx) in enumerate(
    sequence_folds,
    start=1
):

    print(f"\nFold {fold_number}")

    print("Training classes:")
    print(
        pd.Series(
            y_sequences[train_idx]
        ).value_counts().sort_index()
    )

    print("Validation classes:")
    print(
        pd.Series(
            y_sequences[val_idx]
        ).value_counts().sort_index()
    )


Fold 1
Training classes:
0     68
1    190
Name: count, dtype: int64
Validation classes:
0    23
1    44
Name: count, dtype: int64

Fold 2
Training classes:
0     75
1    184
Name: count, dtype: int64
Validation classes:
0    16
1    50
Name: count, dtype: int64

Fold 3
Training classes:
0     77
1    182
Name: count, dtype: int64
Validation classes:
0    14
1    52
Name: count, dtype: int64

Fold 4
Training classes:
0     77
1    182
Name: count, dtype: int64
Validation classes:
0    14
1    52
Name: count, dtype: int64

Fold 5
Training classes:
0     67
1    198
Name: count, dtype: int64
Validation classes:
0    24
1    36
Name: count, dtype: int64


In [18]:
from sklearn.preprocessing import StandardScaler

print("Temporal input shape:")
print(X_sequences.shape)

print(
    "\nExpected format:",
    "(sequences, time_steps, features)"
)

Temporal input shape:
(325, 32, 11)

Expected format: (sequences, time_steps, features)


In [19]:
def create_temporal_summary(X):

    mean_features = X.mean(axis=1)
    std_features = X.std(axis=1)
    first_features = X[:, 0, :]
    last_features = X[:, -1, :]
    change_features = last_features - first_features

    return np.concatenate(
        [
            mean_features,
            std_features,
            first_features,
            last_features,
            change_features
        ],
        axis=1
    )


X_temporal = create_temporal_summary(
    X_sequences
)

print("Original sequence shape:")
print(X_sequences.shape)

print("\nTemporal summary shape:")
print(X_temporal.shape)

Original sequence shape:
(325, 32, 11)

Temporal summary shape:
(325, 55)


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

temporal_results = []

for fold_number, (train_idx, val_idx) in enumerate(
    sequence_folds,
    start=1
):

    X_train = X_temporal[train_idx]
    X_val = X_temporal[val_idx]

    y_train = y_sequences[train_idx]
    y_val = y_sequences[val_idx]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    results = {
        "Fold": fold_number,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, y_pred, zero_division=0
        ),
        "AUROC": roc_auc_score(
            y_val, y_prob
        )
    }

    temporal_results.append(results)

    print(
        f"Fold {fold_number}: "
        f"Accuracy={results['Accuracy']:.4f}, "
        f"F1={results['F1']:.4f}, "
        f"AUROC={results['AUROC']:.4f}"
    )

Fold 1: Accuracy=0.5821, F1=0.7083, AUROC=0.4308
Fold 2: Accuracy=0.4242, F1=0.4865, AUROC=0.4537
Fold 3: Accuracy=0.4697, F1=0.6067, AUROC=0.2761
Fold 4: Accuracy=0.6061, F1=0.7500, AUROC=0.2706
Fold 5: Accuracy=0.6333, F1=0.6562, AUROC=0.6123


In [21]:
temporal_results_df = pd.DataFrame(
    temporal_results
)

print("\nTemporal Logistic Regression")
print("--------------------------------")

print(
    temporal_results_df[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "AUROC"
        ]
    ].agg(["mean", "std"])
)


Temporal Logistic Regression
--------------------------------
      Accuracy  Precision    Recall        F1     AUROC
mean  0.543085   0.726715  0.597058  0.641562  0.408710
std   0.091024   0.041670  0.170802  0.102095  0.141962


In [22]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print(
    "Device:",
    "cuda" if torch.cuda.is_available() else "cpu"
)

PyTorch version: 2.14.0+cpu
Device: cpu


In [23]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


In [24]:
print("X_sequences:", X_sequences.shape)
print("y_sequences:", y_sequences.shape)
print("patient_sequences:", patient_sequences.shape)

X_sequences: (325, 32, 11)
y_sequences: (325,)
patient_sequences: (325,)


In [25]:
class GRUClassifier(nn.Module):

    def __init__(
        self,
        input_size=11,
        hidden_size=32,
        num_layers=1,
        dropout=0.2
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, hidden = self.gru(x)

        last_hidden = hidden[-1]

        last_hidden = self.dropout(
            last_hidden
        )

        return self.fc(last_hidden).squeeze(1)

In [26]:
model = GRUClassifier().to(device)

test_input = torch.tensor(
    X_sequences[:4],
    dtype=torch.float32
).to(device)

test_output = model(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)

Input shape: torch.Size([4, 32, 11])
Output shape: torch.Size([4])


In [27]:
def train_gru(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=100,
    batch_size=32,
    learning_rate=0.001,
    patience=10
):

    train_dataset = TensorDataset(
        torch.tensor(
            X_train,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train,
            dtype=torch.float32
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    positive = np.sum(y_train == 1)
    negative = np.sum(y_train == 0)

    pos_weight = torch.tensor(
        [negative / positive],
        dtype=torch.float32
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32
    ).to(device)

    y_val_tensor = torch.tensor(
        y_val,
        dtype=torch.float32
    ).to(device)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():

            val_logits = model(
                X_val_tensor
            )

            val_loss = criterion(
                val_logits,
                y_val_tensor
            ).item()

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                key: value.cpu().clone()
                for key, value
                in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return model, epoch + 1

In [28]:
from sklearn.preprocessing import StandardScaler

gru_results = []
gru_patient_predictions = []

for fold_number, (train_idx, val_idx) in enumerate(
    sequence_folds,
    start=1
):

    print(f"\nTraining Fold {fold_number}...")

    X_train = X_sequences[train_idx]
    X_val = X_sequences[val_idx]

    y_train = y_sequences[train_idx]
    y_val = y_sequences[val_idx]

    scaler = StandardScaler()

    X_train_2d = X_train.reshape(
        -1,
        X_train.shape[-1]
    )

    X_val_2d = X_val.reshape(
        -1,
        X_val.shape[-1]
    )

    scaler.fit(X_train_2d)

    X_train = scaler.transform(
        X_train_2d
    ).reshape(X_train.shape)

    X_val = scaler.transform(
        X_val_2d
    ).reshape(X_val.shape)

    model = GRUClassifier(
        input_size=11,
        hidden_size=32,
        num_layers=1,
        dropout=0.2
    ).to(device)

    model, epochs_used = train_gru(
        model,
        X_train,
        y_train,
        X_val,
        y_val
    )

    model.eval()

    with torch.no_grad():

        val_tensor = torch.tensor(
            X_val,
            dtype=torch.float32
        ).to(device)

        logits = model(val_tensor)

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    fold_result = {
        "Fold": fold_number,
        "Accuracy": accuracy_score(
            y_val,
            predictions
        ),
        "Precision": precision_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "AUROC": roc_auc_score(
            y_val,
            probabilities
        ),
        "Epochs": epochs_used
    }

    gru_results.append(fold_result)

    fold_patient_predictions = pd.DataFrame({
        "patient_id": patient_sequences[val_idx],
        "y_true": y_val,
        "y_prob": probabilities
    })

    fold_patient_predictions["fold"] = fold_number

    gru_patient_predictions.append(
        fold_patient_predictions
    )

    print(
        f"Fold {fold_number}: "
        f"Accuracy={fold_result['Accuracy']:.4f}, "
        f"F1={fold_result['F1']:.4f}, "
        f"AUROC={fold_result['AUROC']:.4f}, "
        f"Epochs={epochs_used}"
    )

gru_patient_predictions = pd.concat(
    gru_patient_predictions,
    ignore_index=True
)


Training Fold 1...
Fold 1: Accuracy=0.7910, F1=0.8108, AUROC=0.8340, Epochs=23

Training Fold 2...
Fold 2: Accuracy=0.6667, F1=0.7179, AUROC=0.7887, Epochs=25

Training Fold 3...
Fold 3: Accuracy=0.2424, F1=0.1667, AUROC=0.2788, Epochs=11

Training Fold 4...
Fold 4: Accuracy=0.6061, F1=0.7234, AUROC=0.5687, Epochs=11

Training Fold 5...
Fold 5: Accuracy=0.6333, F1=0.6667, AUROC=0.6123, Epochs=17


In [29]:
gru_results_df = pd.DataFrame(
    gru_results
)

print("\nGRU — Sequence-Level GroupKFold")
print("--------------------------------")

print(
    gru_results_df[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "AUROC"
        ]
    ].agg(["mean", "std"])
)


GRU — Sequence-Level GroupKFold
--------------------------------
      Accuracy  Precision    Recall        F1     AUROC
mean  0.587906   0.833571  0.520586  0.617099  0.616508
std   0.205681   0.165473  0.241684  0.257071  0.219822


In [30]:
gru_patient_results = []

for fold_number in sorted(
    gru_patient_predictions["fold"].unique()
):

    fold_data = gru_patient_predictions[
        gru_patient_predictions["fold"] == fold_number
    ].copy()

    patient_data = (
        fold_data
        .groupby("patient_id")
        .agg(
            y_true=("y_true", "first"),
            y_prob=("y_prob", "mean")
        )
        .reset_index()
    )

    patient_data["y_pred"] = (
        patient_data["y_prob"] >= 0.5
    ).astype(int)

    result = {
        "Fold": fold_number,
        "Patients": len(patient_data),
        "Accuracy": accuracy_score(
            patient_data["y_true"],
            patient_data["y_pred"]
        ),
        "Precision": precision_score(
            patient_data["y_true"],
            patient_data["y_pred"],
            zero_division=0
        ),
        "Recall": recall_score(
            patient_data["y_true"],
            patient_data["y_pred"],
            zero_division=0
        ),
        "F1": f1_score(
            patient_data["y_true"],
            patient_data["y_pred"],
            zero_division=0
        ),
        "AUROC": roc_auc_score(
            patient_data["y_true"],
            patient_data["y_prob"]
        )
    }

    gru_patient_results.append(result)

    print(
        f"Fold {fold_number}: "
        f"Patients={result['Patients']}, "
        f"Accuracy={result['Accuracy']:.4f}, "
        f"F1={result['F1']:.4f}, "
        f"AUROC={result['AUROC']:.4f}"
    )

gru_patient_results = pd.DataFrame(
    gru_patient_results
)

Fold 1: Patients=9, Accuracy=0.7778, F1=0.8000, AUROC=0.8333
Fold 2: Patients=9, Accuracy=0.6667, F1=0.7273, AUROC=0.8571
Fold 3: Patients=9, Accuracy=0.3333, F1=0.2500, AUROC=0.2857
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.7143
Fold 5: Patients=8, Accuracy=0.6250, F1=0.6667, AUROC=0.6667


In [31]:
print("\nGRU — Patient-Level GroupKFold")
print("--------------------------------")

print(
    gru_patient_results[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "AUROC"
        ]
    ].agg(["mean", "std"])
)


GRU — Patient-Level GroupKFold
--------------------------------
      Accuracy  Precision    Recall        F1     AUROC
mean  0.613889   0.916667  0.539048  0.642634  0.671429
std   0.166782   0.117851  0.228432  0.225107  0.229857


In [32]:
temporal_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Temporal Logistic Regression",
        "GRU"
    ],
    "Accuracy": [
        rf_patient_mean["accuracy"] if "rf_patient_mean" in globals() else np.nan,
        temporal_results_df["Accuracy"].mean(),
        gru_patient_results["Accuracy"].mean()
    ],
    "Precision": [
        rf_patient_mean["precision"] if "rf_patient_mean" in globals() else np.nan,
        temporal_results_df["Precision"].mean(),
        gru_patient_results["Precision"].mean()
    ],
    "Recall": [
        rf_patient_mean["recall"] if "rf_patient_mean" in globals() else np.nan,
        temporal_results_df["Recall"].mean(),
        gru_patient_results["Recall"].mean()
    ],
    "F1": [
        rf_patient_mean["f1"] if "rf_patient_mean" in globals() else np.nan,
        temporal_results_df["F1"].mean(),
        gru_patient_results["F1"].mean()
    ],
    "AUROC": [
        rf_patient_mean["auroc"] if "rf_patient_mean" in globals() else np.nan,
        temporal_results_df["AUROC"].mean(),
        gru_patient_results["AUROC"].mean()
    ]
})

print("Temporal Model Comparison")
print("-------------------------")
display(temporal_comparison)

Temporal Model Comparison
-------------------------


,Model,Accuracy,Precision,Recall,F1,AUROC
0,Random Forest,NaN,NaN,NaN,NaN,NaN
1,Temporal Logistic Regression,0.543085,0.726715,0.597058,0.641562,0.408710
2,GRU,0.613889,0.916667,0.539048,0.642634,0.671429


In [33]:
# ============================================================
# FINAL GRU PATIENT-LEVEL OOF EVALUATION
# ============================================================

all_gru_patient_predictions = []

for fold_number in sorted(
    gru_patient_predictions["fold"].unique()
):

    fold_data = gru_patient_predictions[
        gru_patient_predictions["fold"] == fold_number
    ].copy()

    patient_data = (
        fold_data
        .groupby("patient_id")
        .agg(
            y_true=("y_true", "first"),
            y_prob=("y_prob", "mean")
        )
        .reset_index()
    )

    patient_data["fold"] = fold_number

    patient_data["y_pred"] = (
        patient_data["y_prob"] >= 0.5
    ).astype(int)

    all_gru_patient_predictions.append(
        patient_data
    )

all_gru_patient_predictions = pd.concat(
    all_gru_patient_predictions,
    ignore_index=True
)

print(
    "Total patient-level OOF predictions:",
    len(all_gru_patient_predictions)
)

print(
    "Unique patients:",
    all_gru_patient_predictions["patient_id"].nunique()
)

print(
    "\nDuplicate patient predictions:"
)

print(
    all_gru_patient_predictions[
        "patient_id"
    ].duplicated().sum()
)

Total patient-level OOF predictions: 44
Unique patients: 44

Duplicate patient predictions:
0


In [36]:
# ============================================================
# FINAL GRU PATIENT-LEVEL METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_true_gru = all_gru_patient_predictions["y_true"]
y_prob_gru = all_gru_patient_predictions["y_prob"]
y_pred_gru = all_gru_patient_predictions["y_pred"]

gru_accuracy = accuracy_score(
    y_true_gru,
    y_pred_gru
)

gru_precision = precision_score(
    y_true_gru,
    y_pred_gru,
    zero_division=0
)

gru_recall = recall_score(
    y_true_gru,
    y_pred_gru,
    zero_division=0
)

gru_f1 = f1_score(
    y_true_gru,
    y_pred_gru,
    zero_division=0
)

gru_auroc = roc_auc_score(
    y_true_gru,
    y_prob_gru
)

gru_cm = confusion_matrix(
    y_true_gru,
    y_pred_gru
)

print("FINAL GRU PATIENT-LEVEL RESULTS")
print("--------------------------------")
print(f"Accuracy : {gru_accuracy:.6f}")
print(f"Precision: {gru_precision:.6f}")
print(f"Recall   : {gru_recall:.6f}")
print(f"F1       : {gru_f1:.6f}")
print(f"AUROC    : {gru_auroc:.6f}")

print("\nConfusion Matrix:")
print(gru_cm)

FINAL GRU PATIENT-LEVEL RESULTS
--------------------------------
Accuracy : 0.613636
Precision: 0.894737
Recall   : 0.531250
F1       : 0.666667
AUROC    : 0.747396

Confusion Matrix:
[[10  2]
 [15 17]]


In [39]:
# ============================================================
# FIND RF OOF PREDICTION VARIABLES
# ============================================================

for name in sorted(list(globals().keys())):
    name_lower = name.lower()

    if (
        "oof" in name_lower
        or "rf_prob" in name_lower
        or "rf_pred" in name_lower
        or "rf_result" in name_lower
    ):
        print(name)

In [40]:
# ============================================================
# FIND GRU PATIENT-LEVEL PREDICTION DATA
# ============================================================

import pandas as pd

print("DataFrames containing GRU / prediction information:\n")

for name in sorted(list(globals().keys())):
    try:
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            cols = obj.columns.tolist()
            text = " ".join(cols).lower()

            if (
                "patient_id" in cols and
                (
                    "gru" in name.lower()
                    or "prob" in text
                    or "pred" in text
                    or "y_true" in text
                    or "y_prob" in text
                )
            ):
                print("=" * 70)
                print("Variable:", name)
                print("Shape:", obj.shape)
                print("Columns:", cols)
                print(obj.head(3).to_string(index=False))

    except Exception:
        pass

DataFrames containing GRU / prediction information:

Variable: all_gru_patient_predictions
Shape: (44, 5)
Columns: ['patient_id', 'y_true', 'y_prob', 'fold', 'y_pred']
patient_id  y_true   y_prob  fold  y_pred
     chf01       1 0.756001     1       1
     chf06       1 0.140538     1       0
     chf11       1 0.562511     1       1
Variable: fold_data
Shape: (60, 4)
Columns: ['patient_id', 'y_true', 'y_prob', 'fold']
patient_id  y_true   y_prob  fold
     chf02       1 0.720410     5
     chf02       1 0.722993     5
     chf02       1 0.734303     5
Variable: fold_patient_predictions
Shape: (60, 4)
Columns: ['patient_id', 'y_true', 'y_prob', 'fold']
patient_id  y_true   y_prob  fold
     chf02       1 0.720410     5
     chf02       1 0.722993     5
     chf02       1 0.734303     5
Variable: gru_patient_predictions
Shape: (325, 4)
Columns: ['patient_id', 'y_true', 'y_prob', 'fold']
patient_id  y_true   y_prob  fold
     chf01       1 0.753129     1
     chf01       1 0.755438     1

In [42]:
# ============================================================
# EXPORT FINAL GRU PATIENT-LEVEL OOF RESULTS
# ============================================================

gru_for_analysis = all_gru_patient_predictions[
    [
        "patient_id",
        "y_true",
        "y_prob",
        "y_pred"
    ]
].copy()

gru_for_analysis = gru_for_analysis.rename(
    columns={
        "y_true": "gru_actual",
        "y_prob": "gru_probability",
        "y_pred": "gru_prediction"
    }
)

print("GRU analysis table:")
print(gru_for_analysis.shape)
print(gru_for_analysis.to_string(index=False))

GRU analysis table:
(44, 4)
patient_id  gru_actual  gru_probability  gru_prediction
     chf01           1         0.756001               1
     chf06           1         0.140538               0
     chf11           1         0.562511               1
    chf202           1         0.630838               1
    chf208           1         0.689389               1
    chf209           1         0.491597               0
    chf213           0         0.337569               0
    chf218           0         0.354606               0
    chf220           0         0.458751               0
     chf05           1         0.624520               1
     chf10           1         0.623983               1
     chf15           1         0.809447               1
    chf207           1         0.569171               1
    chf211           0         0.214733               0
    chf219           1         0.469490               0
    chf224           0         0.356652               0
    chf225          

In [43]:
gru_for_analysis.to_csv(
    "gru_patient_oof_for_analysis.csv",
    index=False
)

print("Saved: gru_patient_oof_for_analysis.csv")

Saved: gru_patient_oof_for_analysis.csv
